# 第7回　データの整理と比較：クロス集計
## ―― 全体で見ると差があるのに、グループごとに見ると差が消えるのはなぜか

情報活用　／　北星学園大学　2026年度後期

今日つくる表は **クロス集計表（分割表）**。2つの項目を同時に数えた表である。
これはこの授業の中核であり、あなたの報告書の中心になる表でもある。

In [ ]:
# 準備：ライブラリと、霊長類376種のデータを読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # 日本語が豆腐（□）にならないようにする

def _build_from_source(keep_missing_code=False):
    """公開データ（PanTHERIA）から、この授業で使う形に組み立て直す。"""
    SRC = "https://esapubs.org/archive/ecol/E090/184/PanTHERIA_1-0_WR05_Aug2008.txt"
    fam = {"Cercopithecidae":"オナガザル科","Cebidae":"オマキザル科","Pitheciidae":"サキ科",
           "Atelidae":"クモザル科","Cheirogaleidae":"コビトキツネザル科","Lemuridae":"キツネザル科",
           "Galagidae":"ガラゴ科","Hylobatidae":"テナガザル科","Indriidae":"インドリ科",
           "Lorisidae":"ロリス科","Lepilemuridae":"イタチキツネザル科","Aotidae":"ヨザル科",
           "Hominidae":"ヒト科","Tarsiidae":"メガネザル科","Daubentoniidae":"アイアイ科"}
    cols = {"MSW05_Binomial":"学名","MSW05_Genus":"属","5-1_AdultBodyMass_g":"体重g",
            "13-1_AdultHeadBodyLen_mm":"頭胴長mm","5-3_NeonateBodyMass_g":"新生児体重g",
            "10-2_SocialGrpSize":"集団サイズ","9-1_GestationLen_d":"妊娠期間日",
            "25-1_WeaningAge_d":"離乳日齢","3-1_AgeatFirstBirth_d":"初産日齢",
            "14-1_InterbirthInterval_d":"出産間隔日","15-1_LitterSize":"一腹産子数",
            "17-1_MaxLongevity_m":"最長寿命月","22-1_HomeRange_km2":"行動圏km2",
            "21-1_PopulationDensity_n/km2":"個体群密度","26-1_GR_Area_km2":"分布域km2",
            "6-2_TrophicLevel":"栄養段階","12-1_HabitatBreadth":"生息環境幅",
            "28-2_Temp_Mean_01degC":"平均気温01","28-1_Precip_Mean_mm":"月降水量mm"}
    src = pd.read_csv(SRC, sep="\t")
    p = src[src["MSW05_Order"] == "Primates"]
    out = p[list(cols)].rename(columns=cols)
    out.insert(1, "科", p["MSW05_Family"].map(fam))
    t = out.pop("平均気温01")
    out["平均気温C"] = np.where(t == -999, -999, (t / 10).round(1))
    out = out.sort_values("学名").reset_index(drop=True)
    return out if keep_missing_code else out.replace(-999, np.nan)

try:
    df = pd.read_csv("https://aonoa68.github.io/joho-katsuyo/data/primates.csv")
except Exception:
    df = _build_from_source()

print("種数:", len(df), " 科数:", df["科"].nunique())
df.head()

---
## 0. なぜピボットテーブルを使わないのか

Excelには「ピボットテーブル」という、同じ表を作る便利な機能がある。**それは使わない。**
作る表は同じである。使わないのは機能のほうだ。理由は1つ。

> **何をどう数えたかが記録されないため、あとから再現できず、他人も検証できない。**

ピボットテーブルは、マウスで列をドラッグして作る。半年後に「この表、どうやって作ったんですか」と聞かれても、**操作の履歴はどこにも残っていない**。
コードなら、どの列を、どの条件で、どう数えたかが全部残る。もう一度実行すれば同じ表が出る。

**結果だけが出てきて過程が残らない道具は、検証に使えない。**
これは、AIの出力に対してこの授業がとる態度と、まったく同じ構造である。

---
## 1. 1973年、カリフォルニア大学バークレー校

大学院の合格率に男女差がある、と大問題になった事件がある。まず全体の数字を見る。

In [ ]:
# 1973年 バークレー校 大学院 出願者の多い6学部の実データ
# 出典: Bickel, Hammel & O'Connell (1975) Science, 187(4175), 398-404.
berkeley = pd.DataFrame({
    "学部": ["A","A","B","B","C","C","D","D","E","E","F","F"],
    "性別": ["男","女"]*6,
    "出願": [825,108, 560,25, 325,593, 417,375, 191,393, 373,341],
    "合格": [512, 89, 353,17, 120,202, 138,131,  53, 94,  22, 24],
})
berkeley["不合格"] = berkeley["出願"] - berkeley["合格"]

total = berkeley.groupby("性別")[["出願","合格"]].sum()
total["合格率"] = (total["合格"] / total["出願"] * 100).round(1)
total

**男性 44.5%、女性 30.4%。** 14ポイントもの差がある。

（よく引用される「男44%・女35%」は大学全体の数字。ここで見ているのは出願者の多い上位6学部で、差はさらに大きい。）

これだけ見れば「女性が不利に扱われている」と読める。ところが、**学部ごとに分けて**数え直すと、話が変わる。

In [ ]:
by_dept = berkeley.pivot_table(index="学部", columns="性別", values=["出願","合格"], aggfunc="sum")
rate = (by_dept["合格"] / by_dept["出願"] * 100).round(1)
rate["女が高い?"] = np.where(rate["女"] > rate["男"], "← 女性のほうが高い", "")
rate

**6学部中4学部で、女性のほうが合格率が高い。** 残り2つも差はわずかである。

全体では女性が14ポイント低いのに、どの学部を見ても女性が不利ではない。なぜこうなるのか。

In [ ]:
apply = by_dept["出願"].copy()
apply["学部全体の合格率"] = (by_dept["合格"].sum(axis=1) / by_dept["出願"].sum(axis=1) * 100).round(1)
apply["女性の出願割合"] = (apply["女"] / (apply["男"] + apply["女"]) * 100).round(1)
apply.sort_values("学部全体の合格率", ascending=False)

答えが出た。**合格率の高い学部A・Bには男性が集中し、合格率の低い学部C・E・Fに女性が集中している。**

学部Aの合格率は64%、学部Fは6%。**そもそも入りやすさが10倍違う。**
女性は難関の学部に多く出願していたので、全体をまぜると合格率が下がって見えた。

> **全体の数字は「差別」、分けると「逆」。どちらが本当かは、分けて初めてわかる。**

これを **シンプソンのパラドックス** という。

### なぜ起こるのか（ひとことで）

```
グループごとに「件数の偏り」や「別の要因」があると、
全体の平均は、その偏りに引っぱられて本当の傾向を隠す（時に逆転させる）
```

平均や全体は、**たった1個にまとめた数字**である。まとめる過程で、グループの情報が消える。

### 数字を見たときの合言葉

1. これ、**何かで分けたら違う話にならないか**（性別・学年・地域・時期・グループ）
2. どこかの**グループが極端に多く／少なく**混じっていないか

---
## 2. クロス集計表を自分で作る

使う関数は `pd.crosstab`。**行に置く項目**と**列に置く項目**を渡すだけ。

霊長類データで「**科によって食べ物が違うのか**」を見る。栄養段階は 1＝草食、2＝雑食 で記録されている。

In [ ]:
d = df.copy()
d["食性"] = d["栄養段階"].map({1: "草食", 2: "雑食"})

# 種数の多い科だけに絞る（少なすぎる科は比較にならない）
big = d["科"].value_counts()
big = big[big >= 15].index
d = d[d["科"].isin(big)]

pd.crosstab(d["科"], d["食性"])

件数の表ができた。ただし **件数のままでは比べられない。**
科ごとに種数が違うので、「オナガザル科は草食が多い」と言っても、単にオナガザル科の種数が多いだけかもしれない。

**割合に直す。** `normalize="index"` で、行ごとに合計100%にする。

In [ ]:
tab = pd.crosstab(d["科"], d["食性"], normalize="index") * 100
tab.round(1)

**オマキザル科とコビトキツネザル科は、記録のある種がすべて雑食。**一方クモザル科は7割が草食（葉や果実を食べる）。科によってはっきり違う。

In [ ]:
# 合計も一緒に出す（報告書にはこの形で載せる）
pd.crosstab(d["科"], d["食性"], margins=True, margins_name="合計")

> **どちらの方向で割ったかを必ず書く。**
> `normalize="index"` は行方向（科ごとに100%）、`normalize="columns"` は列方向（草食の中で100%）。
> 同じ表から、まったく違う主張が作れてしまう。

> **そして、この表の分母は「栄養段階が記録されている種」だけである。**
> 第6回で見たとおり、栄養段階は376種中178種（47%）しか埋まっていない。
> **記録のない種が、どちらかに偏っていない保証はない。**

---
## 3. 層別を1つ加える ―― 結論は変わるか

バークレーでやったのと同じことをする。**3つめの項目で分けてみる。**

「体の大きい霊長類ほど大きな集団で暮らす」というのはよく言われる。まず分けずに見る。

In [ ]:
# 体の大きさで2つに分ける（中央値で区切る）
e = df.dropna(subset=["体重g", "集団サイズ"]).copy()
e["体格"] = np.where(e["体重g"] >= e["体重g"].median(), "大きい", "小さい")

print("【全体】体格別の平均集団サイズ")
print(e.groupby("体格")["集団サイズ"].agg(["count", "mean"]).round(1))

In [ ]:
# 次に、科で分けて見る
print("【科で分けた場合】")
split = e.pivot_table(index="科", columns="体格", values="集団サイズ", aggfunc="mean").round(1)
split["種数"] = e.groupby("科").size()
split[split["種数"] >= 10]

**自分の目で確かめること。** 全体で見えた差は、科ごとに分けても同じ向きに残っているか。消えているか。逆転しているか。

このデータでは、バークレーのような**きれいな逆転は起きない**。**それも結果である。**「分けても結論は変わらなかった」は、報告書に書く価値のある一文だ。確かめずに「差がある」と書くことだけが、してはいけないことである。

---
## 4. 自分の調査データでやる

In [ ]:
# ここから先は「自分の調査データ」でやる。
# Google Forms の回答 → スプレッドシート → ファイル → ダウンロード → CSV で書き出したものを使う。
#
# 左のフォルダアイコン（📁）にCSVをドラッグしてから、ファイル名を書き換えて実行する。
# ※ 数字を手で打ち直さないこと。転記した瞬間に、それは元データではなくなる。

# mydf = pd.read_csv("自分のファイル名.csv")
# mydf.head()

In [ ]:
# クロス集計（列名を自分のものに書き換える）
# pd.crosstab(mydf["行にする項目"], mydf["列にする項目"], margins=True)

# 割合にする
# (pd.crosstab(mydf["行にする項目"], mydf["列にする項目"], normalize="index") * 100).round(1)

---
## 5. 卒業 ―― AIと突き合わせる

同じクロス集計表を **AIに直接作らせて**、自分のコードの結果と一致するか確かめる。

ずれる原因でいちばん多いのは、**空欄の数え方**である。pandasの `crosstab` は空欄の行を自動で除くが、AIが空欄を「その他」として数えていることがある。
**分母が違えば、割合は違う。**

そして、**この表を作らずに同じことが言えるか**も考えてみる。言えるなら、その表はいらない。言えないなら、その表があなたの主張の根拠である。

---
## 課題7（8点）

**このノートブック（コードと出力）** ＋ **解釈3文**。

- [ ] 自分のデータのクロス集計表（件数と割合の両方）
- [ ] 層別を1つ加えた表
- [ ] 解釈3文。**うち1文は「層別したら何が変わったか」**（変わらなかったなら、そう書く）
- [ ] AIの結果と一致したか。ずれた場合は原因

提出期限：次回授業の開始まで（遅れた場合は50%）

---

!!! quote "このデータの出典"
    Jones, K.E. et al. (2009) PanTHERIA: a species-level database of life history,
    ecology, and geography of extant and recently extinct mammals.
    *Ecology* 90(9): 2648. Ecological Archives E090-184.